# ACS Analysis

##Pulling Race/Ethnicity Census Data

This section pulls decennial census data for 2000, 2010, and 2020 and ACS 5-year Estimates from 2009 - 2024. The second ACS function was reworked several times to distinguish between white, hispanic, and Middle Eastern/North African (MENA). Thus, the function draws on B03002 data for mutually exclusive racial groups and contains MENA ancestry information until 2018.

import pandas as pd
import requests
import os
import time

#Set configs for future functions
CENSUS_API_KEY = 'API_KEY_HERE'
STATE_FIPS = '36'  # New York. Change if you need a different state.

os.makedirs(DRIVE_PATH, exist_ok=True)

**Decennial Census Data**

In [ ]:
#Pull tract-level race data from the decennial census. Uses P3 for 2000 and 2010 (SF1), P1 for 2020 (PL 94-171).
#Note: 2020 Decennial does not have a separate MENA category — MENA respondents were classified as White.
#MENA estimates come from ACS ancestry (B04006, 2009-2018) and place-of-birth (B05006) tables instead.

def fetch_decennial_race(year, state_fips=STATE_FIPS, api_key=CENSUS_API_KEY):
    configs = {
        2000: {
            'url': 'https://api.census.gov/data/2000/dec/sf1',
            'vars': ['P003001', 'P003002', 'P003003', 'P003004',
                     'P003005', 'P003006', 'P003007', 'P003008'],
        },
        2010: {
            'url': 'https://api.census.gov/data/2010/dec/sf1',
            'vars': ['P003001', 'P003002', 'P003003', 'P003004',
                     'P003005', 'P003006', 'P003007', 'P003008'],
        },
        2020: {
            'url': 'https://api.census.gov/data/2020/dec/pl',
            'vars': ['P1_001N', 'P1_003N', 'P1_004N', 'P1_005N',
                     'P1_006N', 'P1_007N', 'P1_008N', 'P1_009N'],
        },
    }

    cfg = configs[year]
    params = {
        'get': ','.join(cfg['vars']),
        'for': 'tract:*',
        'in': f'state:{state_fips} county:*',
        'key': api_key,
    }

    r = requests.get(cfg['url'], params=params, timeout=60)
    r.raise_for_status()
    rows = r.json()

    df = pd.DataFrame(rows[1:], columns=rows[0])

    rename_map = dict(zip(cfg['vars'],
                          ['total_pop', 'white', 'black', 'aian',
                           'asian', 'nhpi', 'other', 'two_or_more']))
    df = df.rename(columns=rename_map)

    for col in rename_map.values():
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df['year'] = year
    df['geoid'] = df['state'] + df['county'] + df['tract']

    return df[['geoid', 'state', 'county', 'tract', 'year',
               'total_pop', 'white', 'black', 'aian', 'asian',
               'nhpi', 'other', 'two_or_more']]

In [ ]:
#Pull data for years 2000, 2010, 2020

for year in [2000, 2010, 2020]:
        print(f'Fetching decennial {year}...')
        try:
            df = fetch_decennial_race(year)
            path = os.path.join(DRIVE_PATH, f'race_decennial_{year}.csv')
            df.to_csv(path, index=False)
            print(f'  ✓ Saved {path} ({len(df)} rows)')
        except Exception as e:
            print(f'  ✗ Failed for {year}: {e}')
        time.sleep(0.5)

**ACS Data**

In [ ]:
def fetch_acs_race(year, state_fips=STATE_FIPS, api_key=CENSUS_API_KEY):
    """
    Pull tract-level race/ethnicity from B03002 (Hispanic origin × race),
    plus MENA ancestry from B04006 where available (≤ 2018 5-year ACS).

    Categories returned are mutually exclusive at the race-by-ethnicity level:
      - nh_white, nh_black, nh_aian, nh_asian, nh_nhpi, nh_other,
        nh_two_or_more  (all "Not Hispanic or Latino, ___ alone/combo")
      - hispanic        (Hispanic or Latino, any race)

    MENA ancestry is reported separately and overlaps with nh_white (mostly).
    `nh_white_non_mena` is provided as a lower-bound estimate of non-Hispanic
    white excluding MENA-identifying respondents.
    """
    url = f'https://api.census.gov/data/{year}/acs/acs5'

    # B03002 — Hispanic or Latino Origin by Race
    race_vars = {
        'B03002_001E': 'total_pop',
        'B03002_003E': 'nh_white',
        'B03002_004E': 'nh_black',
        'B03002_005E': 'nh_aian',
        'B03002_006E': 'nh_asian',
        'B03002_007E': 'nh_nhpi',
        'B03002_008E': 'nh_other',
        'B03002_009E': 'nh_two_or_more',
        'B03002_012E': 'hispanic',  # Hispanic or Latino, any race
    }

    # B04006 — ancestry (discontinued after 2018 5-year)
    mena_vars = {
        'B04006_006E': 'anc_arab',     # Arab parent total
        'B04006_048E': 'anc_iranian',
        'B04006_050E': 'anc_israeli',
        'B04006_016E': 'anc_armenian',
    }

    has_ancestry = year <= 2018
    vars_to_get = list(race_vars) + (list(mena_vars) if has_ancestry else [])

    params = {
        'get': ','.join(vars_to_get),
        'for': 'tract:*',
        'in': f'state:{state_fips} county:*',
        'key': api_key,
    }

    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    rows = r.json()

    df = pd.DataFrame(rows[1:], columns=rows[0])
    df = df.rename(columns={**race_vars, **(mena_vars if has_ancestry else {})})

    # Numeric conversion
    for col in race_vars.values():
        df[col] = pd.to_numeric(df[col], errors='coerce')

    if has_ancestry:
        for col in mena_vars.values():
            df[col] = pd.to_numeric(df[col], errors='coerce')
        df['mena_ancestry_total'] = df[list(mena_vars.values())].sum(axis=1, min_count=1)
        df['pct_mena_ancestry'] = df['mena_ancestry_total'] / df['total_pop'] * 100

        # Lower-bound nh_white with MENA removed (overlap assumption: all MENA marked white)
        df['nh_white_non_mena'] = (df['nh_white'] - df['mena_ancestry_total']).clip(lower=0)
    else:
        for col in list(mena_vars.values()) + ['mena_ancestry_total',
                                                'pct_mena_ancestry',
                                                'nh_white_non_mena']:
            df[col] = pd.NA

    df['year'] = year
    df['geoid'] = df['state'] + df['county'] + df['tract']

    return df[[
        'geoid', 'state', 'county', 'tract', 'year',
        'total_pop',
        'nh_white', 'nh_black', 'nh_aian', 'nh_asian',
        'nh_nhpi', 'nh_other', 'nh_two_or_more',
        'hispanic',
        'anc_arab', 'anc_iranian', 'anc_israeli', 'anc_armenian',
        'mena_ancestry_total', 'pct_mena_ancestry',
        'nh_white_non_mena',
    ]]

In [ ]:
for year in range(2009, 2025):  # 2009-2024 inclusive
        print(f'Fetching ACS 5-year {year}...')
        try:
            df = fetch_acs_race(year)
            path = os.path.join(f'/content/drive/MyDrive/NYC Enforcement/src/amelie/Race Overlays/USCB Data/race_acs5_{year}.csv')
            df.to_csv(path, index=False)
            print(f'  ✓ Saved {path} ({len(df)} rows)')
        except Exception as e:
            print(f'  ✗ Failed for {year}: {e}')
        time.sleep(0.5)

## Creating Race/Ethnicity Overlay for Enforcement Clusters

The following cells create race/ethnicity overlays with the following steps:

**1. Loading NYC Tract Shapefiles:** This was done for the 2010 and 2020 NYC tracts

**2. Merging Race Data on Tract Geometry:** Using ACS data for 2024 (most updated) and 2018 (includes more detailed race/ethnicity categories)

**3. Building Coordinate Geodataframes:** Transforming the exact coordinate collections to geodataframes

**4. Merging Points to Tracts:** The coordinate points are each assigned a GEOID

**5. Aggregate point counts to tracts and compute per-capita rates:** The created dataframes contain race/ethnicity and violation/respondent data

**6. Creating Race/Ethnicity Overlays**

In [ ]:
#Setup

!pip install census
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
from census import Census
import os
c = Census("API_KEY_HERE")


ACS_YEAR = 2024
TRACT_YEAR = 2020

**Step 1: Loading NYC Tract Shapefile**

Two different versions of a static race/ethnicity will be created: one from 2023 and one from 2018. This is due to the presence of MENA variables until 2018. This requires two different tract shapefiles to be used, one from 2010 and one from 2020, due to redrawing of tracts every ten years.

In [ ]:
import geopandas as gpd

#Prepare NYC Tract Shapefile (Census tracts clipped to shoreline, water areas not included)

tracts_nyc_2020 = gpd.read_file("/content/drive/MyDrive/NYC Enforcement/src/amelie/Race Overlays/NYC Shapefiles/2020 Tracts/nyct2020.shp")
tracts_nyc_2020['GEOID'] = tracts_nyc_2020['GEOID'].astype(str) #Treating GEOIDs as strings ensures accurate merging


tracts_nyc_2010 = gpd.read_file("/content/drive/MyDrive/NYC Enforcement/src/amelie/Race Overlays/NYC Shapefiles/2010 Tracts/nyct2010.shp")

#Creating GEOIDS and transforming to strings
tracts_nyc_2010['BoroCode'] = pd.to_numeric(tracts_nyc_2010['BoroCode'], errors='coerce')

BORO_TO_COUNTY_FIPS = { # Map NYC BoroCode to federal county FIPS
    1: '061',  # Manhattan / New York County
    2: '005',  # Bronx
    3: '047',  # Brooklyn / Kings County
    4: '081',  # Queens
    5: '085',  # Staten Island / Richmond County
}

# Build GEOID: state FIPS (36) + county FIPS + tract code
tracts_nyc_2010['GEOID'] = (
    '36'
    + tracts_nyc_2010['BoroCode'].map(BORO_TO_COUNTY_FIPS)
    + tracts_nyc_2010['CT2010'].astype(str).str.zfill(6)
)
tracts_nyc_2010['GEOID'] = tracts_nyc_2010['GEOID'].astype(str)


#Reprojecting to EPSG:2263 (feet) for accurate area/distance calculations
tracts_nyc_2020 = tracts_nyc_2020.to_crs(epsg = 2263)
tracts_nyc_2010 = tracts_nyc_2010.to_crs(epsg = 2263)

**Step 2: Merging Race Data on Tract Geometry**

The following cells build two dataframes, `tracts_2024` and `tracts_2018` that merges the race data onto the tract geometries.

In [ ]:
# Function to merge demographics + compute percentages

def build_tract_demographics(race_csv_path, tract_gdf, include_mena=False):
    """
    Load a race CSV (output from fetch_acs_race), merge to tract geometry, and compute percentage columns.

    Parameters
    ----------
    race_csv_path : str
        Path to the saved race CSV.
    tract_gdf : GeoDataFrame
        NYC tract geometry (already filtered + reprojected).
    include_mena : bool
        If True, compute MENA percentage columns. Use for years <= 2018.
    """
    race_nyc = pd.read_csv(race_csv_path, dtype={'geoid': str})

    # Drop zero-population tracts (parks, cemeteries, airports)
    race_nyc = race_nyc[race_nyc['total_pop'] > 0].copy()

    # Compute race percentages
    race_nyc['pct_white']       = race_nyc['nh_white']       / race_nyc['total_pop'] * 100
    race_nyc['pct_nh_white_non_mena'] = race_nyc['nh_white_non_mena'] / race_nyc['total_pop'] * 100
    race_nyc['pct_black']       = race_nyc['nh_black']       / race_nyc['total_pop'] * 100
    race_nyc['pct_aian']        = race_nyc['nh_aian']        / race_nyc['total_pop'] * 100
    race_nyc['pct_asian']       = race_nyc['nh_asian']       / race_nyc['total_pop'] * 100
    race_nyc['pct_nhpi']        = race_nyc['nh_nhpi']        / race_nyc['total_pop'] * 100
    race_nyc['pct_other']       = race_nyc['nh_other']       / race_nyc['total_pop'] * 100
    race_nyc['pct_two_or_more'] = race_nyc['nh_two_or_more'] / race_nyc['total_pop'] * 100
    race_nyc['pct_hispanic']    = race_nyc['hispanic']       / race_nyc['total_pop'] * 100
    race_nyc['pct_nonwhite']    = 100 - race_nyc['pct_white']
    race_nyc['pct_nonwhite_broad'] = 100 - race_nyc['pct_nh_white_non_mena']

    # MENA percentage (only for years with B04006 ancestry data)
    if include_mena:
        # pct_mena_ancestry was already computed in fetch_acs_race, but recompute
        # in case the CSV was saved before that aggregation existed
        if 'pct_mena_ancestry' not in race_nyc.columns:
            mena_cols = ['anc_arab', 'anc_iranian', 'anc_israeli', 'anc_armenian']
            race_nyc['mena_ancestry_total'] = race_nyc[mena_cols].sum(axis=1, min_count=1)
            race_nyc['pct_mena_ancestry'] = (
                race_nyc['mena_ancestry_total'] / race_nyc['total_pop'] * 100
            )

    # Merge to tract geometry
    merged = tract_gdf.merge(
        race_nyc,
        left_on='GEOID',
        right_on='geoid',
        how='left'
    )

    # Diagnostic: how many tracts didn't get demographic data?
    unmatched = merged['total_pop'].isna().sum()
    print(f'  {unmatched} tracts without demographic data '
          f'({unmatched/len(merged)*100:.1f}%)')

    return merged

In [ ]:
# Version 1: 2024 (most current demographics, no MENA)
print('Building 2024 version...')
tracts_2024 = build_tract_demographics(
    race_csv_path=os.path.join(DRIVE_PATH + '/Race Overlays/USCB Data/race_acs5_2024.csv'),
    tract_gdf=tracts_nyc_2020,
    include_mena=False
)
print(f'  {len(tracts_2024)} tracts in merged GeoDataFrame\n')
tracts_2024.to_file(DRIVE_PATH + "Race Overlays/tracts_2024.geojson", driver="GeoJSON")

# Version 2: 2018 (last year with ancestry/MENA data)
print('Building 2018 version...')
tracts_2018 = build_tract_demographics(
    race_csv_path=os.path.join(DRIVE_PATH + '/Race Overlays/USCB Data/race_acs5_2018.csv'),
    tract_gdf=tracts_nyc_2010,
    include_mena=True
)
print(f'  {len(tracts_2018)} tracts in merged GeoDataFrame\n')
tracts_2018.to_file(DRIVE_PATH + "Race Overlays/tracts_2018.geojson", driver = "GeoJSON")

**Step 3: Building Coordinate Geodataframes**

The following cells draw on `violation_exact_coordinates` and `respondent_exact_coordinates` to create geodataframes for future merging

In [ ]:
import pandas as pd
import geopandas as gpd

# Converting to GDF
violation_exact_coordinates_gdf = gpd.GeoDataFrame(
    violation_exact_coordinates,
    geometry=gpd.points_from_xy(violation_exact_coordinates['lon'], violation_exact_coordinates['lat']),
    crs='EPSG:4326'
)

respondent_exact_coordinates_gdf = gpd.GeoDataFrame(
    respondent_exact_coordinates,
    geometry = gpd.points_from_xy(respondent_exact_coordinates['lon'], respondent_exact_coordinates['lat']),
    crs = 'EPSG:4326'
)

#Explicitly reprojecting
violation_exact_coordinates_gdf = violation_exact_coordinates_gdf.to_crs("EPSG: 2263")
respondent_exact_coordinates_gdf = respondent_exact_coordinates_gdf.to_crs("EPSG: 2263")

violation_exact_coordinates_gdf.to_file(DRIVE_PATH + "Race Overlays/violation_exact_coordinates_gdf.geojson", driver = "GeoJSON")
respondent_exact_coordinates_gdf.to_file(DRIVE_PATH + "Race Overlays/respondent_exact_coordinates_gdf.geojson", driver = "GeoJSON")

**Step 4: Joining Points to Tracts**

The following cells create a function to perform spatial joins for violation and respondent tracts. The coordinates from `violation_exact_coordinates_gdf` and `respondent_exact_coordinates_gdf` are assigned the GEOID of the tract they fall in. This is done for violations and respondents for both 2024 and 2018.

#Spatial join violations and respondents to tracts.
#Inputs are already GeoDataFrames with exact-coordinate geometries.

def spatial_join_to_tracts(points_gdf, tract_gdf, points_label='', version_label=''):
    """
    Spatially join a points GeoDataFrame to a tract GeoDataFrame,
    assigning each point a tract GEOID.

    Parameters
    ----------
    points_gdf : GeoDataFrame
        Points (violations or respondents) with geometry already set.
    tract_gdf : GeoDataFrame
        Tract geometry in EPSG:2263 (NY State Plane) with a 'GEOID' column.
    points_label : str
        For diagnostic print (e.g. 'violations', 'respondents').
    version_label : str
        For diagnostic print (e.g. '2024' or '2018').

    Returns
    -------
    GeoDataFrame
        Original points with an added 'tract_geoid' column.
    """

    if points_gdf.crs != tract_gdf.crs:
        points_gdf = points_gdf.to_crs(tract_gdf.crs)

    # Spatial join to see which tracts points fall in
    joined = gpd.sjoin(
        points_gdf,
        tract_gdf[['GEOID', 'geometry']],
        how='left',
        predicate='within'
    )

    # Clean up join artifacts
    joined = joined.rename(columns={'GEOID': 'tract_geoid'})
    if 'index_right' in joined.columns:
        joined = joined.drop(columns=['index_right'])

    # Diagnostic: how many points didn't land in any tract?
    n_total = len(joined)
    n_unjoined = joined['tract_geoid'].isna().sum()
    print(f'[{version_label} {points_label}] {n_total:,} points total, '
          f'{n_unjoined:,} not in any tract ({n_unjoined/n_total*100:.1f}%)')

    return joined

In [ ]:
print('Joining to 2020-vintage tracts (for 2024 analysis)...')
viol_tract_2024 = spatial_join_to_tracts(
    violation_exact_coordinates_gdf, tracts_nyc_2020,
    points_label='violations', version_label='2024'
)
resp_tract_2024 = spatial_join_to_tracts(
    respondent_exact_coordinates_gdf, tracts_nyc_2020,
    points_label='respondents', version_label='2024'
)

In [ ]:
print('\nJoining to 2010-vintage tracts (for 2018 analysis)...')
viol_tract_2018 = spatial_join_to_tracts(
    violation_exact_coordinates_gdf, tracts_nyc_2010,
    points_label='violations', version_label='2018'
)
resp_tract_2018 = spatial_join_to_tracts(
    respondent_exact_coordinates_gdf, tracts_nyc_2010,
    points_label='respondents', version_label='2018'
)

**Step 5: Aggregate point counts to tracts and compute per-capita rates**

The following function counts points per tract and merges it back to the tract-level demographic GDF. It computes per 1K resident rate for violations and respondents coordinates located in that tract to have a comparable metric. Metrics are recalculated for 2018 to estimate more accurate values for white people by taking MENA ancestry into account; this is done by subtracting every other race category from the total population.

In [ ]:
def aggregate_to_tracts(points_with_tract, tracts_demo, points_col_name):
    """
    Parameters
    ----------
    points_with_tract : GeoDataFrame
        Output of spatial_join_to_tracts — must have 'tract_geoid' column.
    tracts_demo : GeoDataFrame
        Tract geometry + demographics — must have 'GEOID' and 'total_pop'.
    points_col_name : str
        Name of the count column (e.g. 'violation_count', 'respondent_count').

    Returns
    -------
    GeoDataFrame
        tracts_demo with count and rate columns added.
    """
    # Count points per tract
    counts = (
        points_with_tract.dropna(subset=['tract_geoid']) #Drop unjoined points with no tract geoid
        .groupby('tract_geoid')
        .size()
        .reset_index(name=points_col_name)
    )

    # Merge counts back to the demographic GeoDataFrame
    rate_col_name = points_col_name.replace('_count', '_per_1k')
    out = tracts_demo.merge(counts, left_on='GEOID', right_on='tract_geoid', how='left')
    out[points_col_name] = out[points_col_name].fillna(0)
    out[rate_col_name] = out[points_col_name] / out['total_pop'] * 1000
    out = out.drop(columns=['tract_geoid'])
    return out